In [1]:
import random
import re

# Regex patterns for post-processing
punct_fix_end = re.compile(r"\s+([.,!?;:])$")       # Remove space before ending punctuation
ellipsis_patterns = [
    re.compile(r"\s*\.\s*\.\s*\.\s*"),             # any spaced ... sequences
    re.compile(r"\s*\.\s*\.\s*"),                  # .. sequences
]
quotes_fix = re.compile(r"[“”]\s*(.*?)\s*[“”]", flags=re.UNICODE)  # text between quotes
hyphen_fix = re.compile(r"\b([A-Za-z]+)\s*-\s*([A-Za-z]+)\b")   
hyphen_fix3 = re.compile(r"\b([A-Za-z]+)\s*-\s*([A-Za-z]+)\s*-\s*([A-Za-z]+)\b")     # join split hyphenated words
hyphen_fix4 = re.compile(r"\b([A-Za-z]+)\s*-\s*([A-Za-z]+)\s*-\s*([A-Za-z]+)\s*-\s*([A-Za-z]+)\b") 
hyphen_fix5 = re.compile(r"\b([A-Za-z]+)\s*-\s*([A-Za-z]+)\s*-\s*([A-Za-z]+)\s*-\s*([A-Za-z]+)\s*-\s*([A-Za-z]+)\b") 
clitic_fix = re.compile(r"\b([A-Za-z]+)\s+('re|'m|'s|'ve|'d|'ll|n['’]t)\b", flags=re.IGNORECASE)
INSIDE_BRACKETS = re.compile(r"([\(\[\{])\s*(.*?)\s*([\)\]\}])")
INSIDE_QUOTES = re.compile(r"([\"'‘“”])\s*(.*?)\s*([\"'“’”])")
REMOVE_SPACE_BEFORE_PUNCT = re.compile(r"\s+([,.:;!?])")  
hyphen_fix_pattern = re.compile(r"\b([A-Za-z0-9]+)\s*-\s*([A-Za-z0-9]+)\b")

def normalize_sentence(text: str) -> str:
    # Preserve leading spaces
    leading_space = ""
    if text.startswith(" "):
        leading_space = " "
    
    text = text.strip()

    # Fix clitics (general contractions)
    text = clitic_fix.sub(lambda m: m.group(1) + m.group(2), text)

    # Fix hyphenated words: join split hyphenated words
    text = hyphen_fix.sub(r"\1-\2", text)
    text = hyphen_fix3.sub(r"\1-\2-\3", text)
    text = hyphen_fix4.sub(r"\1-\2-\3-\4", text)
    text = hyphen_fix5.sub(r"\1-\2-\3-\4-\5", text)

    # Fix text inside quotes: “ text ” -> “text”
    text = quotes_fix.sub(r'“\1”', text)
    text = re.sub(r"\s+([.,!?;:])\s*", r"\1 ", text)
    text = re.sub(r"([.,!?;:])\s+\"", r'\1"', text)

    if text.startswith("' "):
        text = text.replace("' ", "'", 1)
    if text.startswith('" '):
        text = text.replace('" ', '"', 1)
    if text.endswith('  . "'):
        text = text.replace('  . "', '."', 1)
    if ' ...' in text:
        text = text.replace(' ...', '...', 1)
    if ' . ..' in text:
        text = text.replace(' . ..', '...', 1)
    if ' . . .' in text:
        text = text.replace(' . . .', '...', 1)

    # Fix ellipses
    for pattern in ellipsis_patterns:
        text = pattern.sub("...", text)

    # Remove spaces before ending punctuation
    text = punct_fix_end.sub(r"\1", text)

    # Collapse multiple spaces inside sentence
    text = re.sub(r"\s+", " ", text).strip()
    text = REMOVE_SPACE_BEFORE_PUNCT.sub(r"\1", text)
    text = INSIDE_BRACKETS.sub(r"\1\2\3", text)
    text = INSIDE_QUOTES.sub(r"\1\2\3", text)
    text = hyphen_fix_pattern.sub(r"\1-\2", text)

    return leading_space + text


def shuffle_1gram(sentence, seed=None):
    """
    Shuffle a sentence at the 1-gram (token) level while keeping punctuation (except apostrophes) in place.
    Ensures that no non-punctuation token remains in its original position.
    
    Args:
        sentence (str): Input sentence.
        seed (int, optional): Random seed for reproducibility.

    Returns:
        str: Shuffled sentence with punctuation restored.
    """
    if seed is not None:
        random.seed(seed)

    # Regex: words including internal apostrophes OR any punctuation except apostrophes
    token_pattern = re.compile(r"\b\w+(?:'\w+)?\b|[^\w\s']") 
    tokens = token_pattern.findall(sentence)

    # Store punctuation positions (except apostrophes inside words)
    punct_positions = {i: tok for i, tok in enumerate(tokens) if re.match(r"[^\w\s']", tok)}

    # Extract only non-punctuation tokens for shuffling
    words = [tok for i, tok in enumerate(tokens) if i not in punct_positions]

    # Edge case: if 1 or 0 words, return original
    if len(words) <= 1:
        return sentence

    # Shuffle until no word remains in its original position
    attempts = 0
    while True:
        shuffled_words = words.copy()
        random.shuffle(shuffled_words)
        same_position_count = sum([w1 == w2 for w1, w2 in zip(words, shuffled_words)])
        attempts += 1
        if same_position_count == 0 or attempts > 100:
            break

    # Reconstruct sentence with punctuation in original positions
    shuffled_tokens = []
    word_idx = 0
    for i in range(len(tokens)):
        if i in punct_positions:
            shuffled_tokens.append(punct_positions[i])
        else:
            shuffled_tokens.append(shuffled_words[word_idx])
            word_idx += 1

    shuffles_tr = " ".join(shuffled_tokens)
    
    shuffled_sentence = normalize_sentence(shuffles_tr)
    
    return shuffled_sentence



## CHILDES

In [ ]:
with open('./corpora/CHILDES_rand/original/childes_rand.train.txt', 'r') as file:
    with open('./corpora/CHILDES_rand/shuffle_order/1gram/childes_rand.train.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            shuffled_line = shuffle_1gram(line.strip(), seed=42)
            outfile.write(shuffled_line + '\n')

In [ ]:
with open('./corpora/CHILDES_rand/original/childes_rand.dev.txt', 'r') as file:
    with open('./corpora/CHILDES_rand/shuffle_order/1gram/childes_rand.dev.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            shuffled_line = shuffle_1gram(line.strip(), seed=42)
            outfile.write(shuffled_line + '\n')

## WIKIPEDIA

In [ ]:
with open('./corpora/WIKI_rand/original/wiki_rand.train.txt', 'r') as file:
    with open('./corpora/WIKI_rand/shuffle_order/1gram/wiki_rand.train.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            shuffled_line = shuffle_1gram(line.strip(), seed=42)
            outfile.write(shuffled_line + '\n')


In [ ]:
with open('./corpora/WIKI_rand/original/wiki_rand.dev.txt', 'r') as file:
    with open('./corpora/WIKI_rand/shuffle_order/1gram/wiki_rand.dev.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            shuffled_line = shuffle_1gram(line.strip(), seed=42)
            outfile.write(shuffled_line + '\n')


## BNC

In [ ]:
with open('./corpora/BNC_rand/original/bnc_rand.train.txt', 'r') as file:
    with open('./corpora/BNC_rand/shuffle_order/1gram/bnc_rand.train.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            shuffled_line = shuffle_1gram(line.strip(), seed=42)
            outfile.write(shuffled_line + '\n')

In [ ]:
with open('./corpora/BNC_rand/original/bnc_rand.dev.txt', 'r') as file:
    with open('./corpora/BNC_rand/shuffle_order/1gram/bnc_rand.dev.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            shuffled_line = shuffle_1gram(line.strip(), seed=42)
            outfile.write(shuffled_line + '\n')

## CANDOR

In [ ]:
with open('./corpora/CANDOR_rand/original/candor_rand.train.txt', 'r') as file:
    with open('./corpora/CANDOR_rand/shuffle_order/1gram/candor_rand.train.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            shuffled_line = shuffle_1gram(line.strip(), seed=42)
            outfile.write(shuffled_line + '\n')

In [ ]:
with open('./corpora/CANDOR_rand/original/candor_rand.dev.txt', 'r') as file:
    with open('./corpora/CANDOR_rand/shuffle_order/1gram/candor_rand.dev.txt', 'w') as outfile:
        lines = file.readlines()
        for line in lines:
            shuffled_line = shuffle_1gram(line.strip(), seed=42)
            outfile.write(shuffled_line + '\n')